# Race Telemetry — Session Explorer

Starter notebook for querying and visualising telemetry sessions from DynamoDB.

**Prerequisites**
```
pip install boto3 pandas matplotlib folium
```
Configure AWS credentials with read access to `telemetry-runs` (ask for your access key):
```
aws configure --profile race-telemetry
```

In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import folium
from decimal import Decimal

AWS_PROFILE = 'race-telemetry'   # change if your profile name differs
AWS_REGION  = 'us-west-2'
TABLE_NAME  = 'telemetry-runs'

session  = boto3.Session(profile_name=AWS_PROFILE)
dynamodb = session.resource('dynamodb', region_name=AWS_REGION)
table    = dynamodb.Table(TABLE_NAME)

## List available sessions

In [ ]:
# Scan for distinct session IDs (fine for small datasets; use query once sessions are large)
result   = table.scan(ProjectionExpression='session_id')
sessions = sorted({row['session_id'] for row in result['Items']})
print('Sessions found:', sessions)

## Load a session

In [ ]:
SESSION_ID = sessions[-1]   # pick the most recent, or set manually e.g. 'S003'
print('Loading session:', SESSION_ID)

from boto3.dynamodb.conditions import Key

items = []
resp  = table.query(KeyConditionExpression=Key('session_id').eq(SESSION_ID))
items.extend(resp['Items'])
while 'LastEvaluatedKey' in resp:
    resp = table.query(
        KeyConditionExpression=Key('session_id').eq(SESSION_ID),
        ExclusiveStartKey=resp['LastEvaluatedKey']
    )
    items.extend(resp['Items'])

df = pd.DataFrame(items)

# Type coercion — DynamoDB returns Decimal for numbers
for col in ['lat', 'lon', 'speed_kph', 'heading']:
    if col in df.columns:
        df[col] = df[col].astype(float)
for col in ['satellites', 'sequence']:
    if col in df.columns:
        df[col] = df[col].astype(int)

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values('timestamp').reset_index(drop=True)
df['elapsed_s'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
df['speed_mph'] = df['speed_kph'] * 0.621371

print(f'{len(df)} records  |  {df["elapsed_s"].max():.0f}s elapsed  |  {df["satellites"].max()} max sats')
df.head()

## Speed trace

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['elapsed_s'], df['speed_mph'], linewidth=1)
ax.set_xlabel('Elapsed time (s)')
ax.set_ylabel('Speed (mph)')
ax.set_title(f'Session {SESSION_ID} — speed trace')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## GPS map

In [ ]:
center = [df['lat'].mean(), df['lon'].mean()]
m = folium.Map(location=center, zoom_start=15, tiles='CartoDB dark_matter')

coords = list(zip(df['lat'], df['lon']))
folium.PolyLine(coords, color='#58a6ff', weight=2, opacity=0.8).add_to(m)

# Mark start and end
folium.CircleMarker(coords[0],  radius=6, color='green', fill=True, popup='Start').add_to(m)
folium.CircleMarker(coords[-1], radius=6, color='red',   fill=True, popup='End').add_to(m)

m

## Distance accumulation

Switches the X axis from time → distance around the track.
This is the foundation for lap alignment and trace overlay — see issue #9.

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi  = radians(lat2 - lat1)
    dlam  = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlam/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

dists = [0.0]
for i in range(1, len(df)):
    d = haversine_m(df.loc[i-1,'lat'], df.loc[i-1,'lon'],
                    df.loc[i,  'lat'], df.loc[i,  'lon'])
    dists.append(dists[-1] + d)

df['distance_m'] = dists

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['distance_m'], df['speed_mph'], linewidth=1)
ax.set_xlabel('Distance from session start (m)')
ax.set_ylabel('Speed (mph)')
ax.set_title(f'Session {SESSION_ID} — speed vs distance')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Total distance: {df["distance_m"].max():.0f} m  ({df["distance_m"].max()/1609:.2f} miles)')

## Lap detection stub

Define the start/finish line as two GPS points (record these on-site at the track).
Detect when the car's path crosses the line segment.
See issue #9 for discussion on approach.

In [ ]:
# TODO: fill in after recording S/F coords on-site at PIR or ORE
SF_LINE = {
    'p1': {'lat': 0.0, 'lon': 0.0},   # left edge of S/F line
    'p2': {'lat': 0.0, 'lon': 0.0},   # right edge of S/F line
}

def segments_cross(a1, a2, b1, b2):
    """True if line segment a1->a2 crosses b1->b2."""
    def cross(o, a, b):
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    d1 = cross(b1, b2, a1)
    d2 = cross(b1, b2, a2)
    d3 = cross(a1, a2, b1)
    d4 = cross(a1, a2, b2)
    if ((d1>0 and d2<0) or (d1<0 and d2>0)) and ((d3>0 and d4<0) or (d3<0 and d4>0)):
        return True
    return False

p1 = (SF_LINE['p1']['lat'], SF_LINE['p1']['lon'])
p2 = (SF_LINE['p2']['lat'], SF_LINE['p2']['lon'])

lap = 0
lap_nums = [0]
for i in range(1, len(df)):
    a1 = (df.loc[i-1,'lat'], df.loc[i-1,'lon'])
    a2 = (df.loc[i,  'lat'], df.loc[i,  'lon'])
    if segments_cross(a1, a2, p1, p2):
        lap += 1
    lap_nums.append(lap)

df['lap'] = lap_nums
print('Laps detected:', df['lap'].max())
print(df.groupby('lap').agg(duration_s=('elapsed_s', lambda x: x.max()-x.min()), points=('lat','count')))